In [43]:
import rasterio
import xarray as xr
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Read raster as DataArray
with rasterio.open('/home/gisuser/code/data/combined_rc_timeseries/_combined_rc.tif') as src:
    array = src.read(1)  # use src.read() for multi-band
    nodata = src.nodata

# Create an xarray DataArray with spatial dims
da = xr.DataArray(array, dims=['y', 'x'])

# Mask out nodata values
masked_da = da.where(da != nodata)

# Flatten for clustering (drop nan values)
pixel_vals = masked_da.values.flatten()
pixel_vals = pixel_vals[~np.isnan(pixel_vals)].reshape(-1, 1)

# Run cluster analysis
kmeans = KMeans(n_clusters=4, random_state=1)
labels = kmeans.fit_predict(pixel_vals)

# Map cluster labels back to raster shape
cluster_array = np.full(masked_da.shape, np.nan)
cluster_array[~np.isnan(masked_da.values)] = labels

# Convert to xarray DataArray
cluster_da = xr.DataArray(cluster_array, dims=['y', 'x'])

# Plot results
cluster_da.plot(cmap='tab20')
plt.title('Spatial Cluster Analysis')
plt.show()

# Optional: export clustered raster
with rasterio.open('clustered_raster.tif', 'w', 
                   driver='GTiff',
                   height=cluster_da.shape[0],
                   width=cluster_da.shape[1],
                   count=1,
                   dtype=rasterio.float32,
                   crs=src.crs,
                   transform=src.transform) as dst:
    dst.write(cluster_da.values, 1)


KeyboardInterrupt: 

In [ ]:
import rasterio
import xarray as xr
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Read raster
with rasterio.open('/home/gisuser/code/data/combined_rc_timeseries/_combined_rc.tif') as src:
    array = src.read(1)
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

# Create DataArray and mask nodata
da = xr.DataArray(array, dims=['y', 'x'])
masked_da = da.where(da != nodata)

# Prepare data for clustering
pixel_vals = masked_da.values.flatten()
valid_pixels = pixel_vals[~np.isnan(pixel_vals)].reshape(-1, 1)

# Run clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(valid_pixels)

print(f"Cluster centers: {kmeans.cluster_centers_.flatten()}")


In [ ]:
# Map labels back to raster
cluster_array = np.full(masked_da.shape, np.nan)
cluster_array[~np.isnan(masked_da.values)] = labels
cluster_da = xr.DataArray(cluster_array, dims=['y', 'x'])

# Plot
plt.figure(figsize=(10, 8))
cluster_da.plot(cmap='tab10', add_colorbar=True)
plt.title('K-Means Cluster Analysis (k=4)')
plt.show()

# Export
cluster_export = np.where(np.isnan(cluster_array), -9999, cluster_array)
with rasterio.open('clustered_raster.tif', 'w', 
                   driver='GTiff', height=cluster_da.shape[0], width=cluster_da.shape[1],
                   count=1, dtype=rasterio.int16, crs=crs, transform=transform,
                   nodata=-9999) as dst:
    dst.write(cluster_export.astype(rasterio.int16), 1)

print("Clustering complete. Output saved as 'clustered_raster.tif'")

In [ ]:
# cluster based on local typology pattern
import rasterio
import numpy as np
from sklearn.cluster import KMeans
from scipy.ndimage import generic_filter
import matplotlib.pyplot as plt

# Read single typology raster
with rasterio.open('/home/gisuser/code/data/combined_rc_timeseries/_combined_rc.tif') as src:
    typology = src.read(1)
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

# Create features from typology patterns using moving windows
def extract_features(array, window_size=5):
    def window_stats(window):
        valid = window[window != nodata]
        if len(valid) == 0:
            return [0, 0, 0]  # mean, std, dominant_class
        return [np.mean(valid), np.std(valid), np.bincount(valid.astype(int)).argmax()]
    
    # Extract features for each pixel
    features = []
    for i in range(3):  # mean, std, dominant_class
        feature_map = generic_filter(array, lambda x: window_stats(x)[i], size=window_size)
        features.append(feature_map.flatten())
    
    return np.column_stack(features)

# Extract spatial features
features = extract_features(typology)

# Remove nodata pixels
valid_mask = typology.flatten() != nodata
valid_features = features[valid_mask]

# Cluster by typology patterns
kmeans = KMeans(n_clusters=5, random_state=42)
labels = kmeans.fit_predict(valid_features)

# Map back to raster
cluster_array = np.full(typology.size, -9999)
cluster_array[valid_mask] = labels
cluster_map = cluster_array.reshape(typology.shape)

# Plot
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.imshow(typology, cmap='tab10')
plt.title('Original Typologies')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(cluster_map, cmap='viridis')
plt.title('Typology Pattern Clusters')
plt.colorbar()
plt.show()


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f6694f4fa70>>
Traceback (most recent call last):
  File "/opt/conda/envs/base_gis/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 796, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/base_gis/lib/python3.12/threading.py", line 1543, in enumerate
    with _active_limbo_lock:
         ^^^^^^^^^^^^^^^^^^
SystemError: <built-in method __enter__ of _thread.RLock object at 0x7f669790e140> returned a result with an exception set


In [ ]:
import rasterio
import xarray as xr
import numpy as np
from sklearn.cluster import KMeans
from sklearn.utils import shuffle
import matplotlib.pyplot as plt

def process_chunk_for_kmeans(chunk, ncluster):
    DIFFcluster = shuffle(chunk.stack(z=("x", "y")).data.reshape(-1, 1), random_state=0, n_samples=1_000)
    kmeans = KMeans(n_clusters=ncluster, random_state=0, n_init=10).fit(DIFFcluster)
    return kmeans.cluster_centers_

# Read raster
with rasterio.open('/home/gisuser/code/data/combined_rc_timeseries/_combined_rc.tif') as src:
    array = src.read(1)
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

# Create DataArray and mask nodata
da = xr.DataArray(array, dims=['y', 'x'])
masked_da = da.where(da != nodata)

# Get cluster centers using your function
ncluster = 5
cluster_centers = process_chunk_for_kmeans(masked_da, ncluster)
print(f"Cluster centers: {cluster_centers.flatten()}")

# Apply clustering to full dataset
pixel_vals = masked_da.values.flatten()
valid_pixels = pixel_vals[~np.isnan(pixel_vals)].reshape(-1, 1)

# Use pre-computed centers to predict labels
kmeans = KMeans(n_clusters=ncluster, init=cluster_centers, n_init=1, random_state=0)
labels = kmeans.fit_predict(valid_pixels)

# Map back to raster
cluster_array = np.full(masked_da.shape, np.nan)
cluster_array[~np.isnan(masked_da.values)] = labels
cluster_da = xr.DataArray(cluster_array, dims=['y', 'x'])

# Plot
plt.figure(figsize=(10, 8))
cluster_da.plot(cmap='tab10')
plt.title(f'K-Means Clustering (k={ncluster})')
plt.show()




# Function to perform KMeans clustering on a chunk
    def process_chunk_for_kmeans(chunk, ncluster):
        DIFFcluster = shuffle(chunk.stack(z=("x", "y")).data.reshape(-1, 1), random_state=0, n_samples=1_000)
        kmeans = KMeans(n_clusters=ncluster, random_state=0, n_init=10).fit(DIFFcluster)
        return kmeans.cluster_centers_